# Projeto de Controladores

In [22]:
# Bibliotecas utilizadas
import matplotlib.pyplot as plt
import sympy as sp
from sympy import symbols, I, Abs, re, im, Poly
from sympy.physics.control import *
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath
from collections import Counter
import plotly.graph_objects as go

In [23]:
# Símbolos utilizados
s, k = sp.symbols('s k')

## Funções auxiliares

In [24]:
def polinomio(entrada):
    '''
    Função que transforma coeficientes em polinômios
    '''
    coeficientes = entrada.strip().split()  # Transforma a entrada em uma lista de strings e remove espaços extras
    coeficientes.reverse() # Inverte a lista
    lista_polinomio = []

    # Percorre de trás para frente para começar pelo maior expoente
    for exp in range(len(coeficientes) - 1, -1, -1):
        coef = coeficientes[exp]

        # Pula coeficientes que são zero
        if coef == "0":
            continue

        # Formata o termo dependendo do expoente
        if exp > 1:
            termo = f"{coef}*s**{exp}"
        elif exp == 1:
            termo = f"{coef}*s"
        else:
            termo = f"{coef}"

        lista_polinomio.append(termo)

    return " + ".join(lista_polinomio)


In [25]:
def definir_controlador():
    PI = False
    PD = False
    PID = False
    controlador = input("Forneça o tipo de controlador").upper() 
    while(controlador != "PI" and controlador != "PD" and controlador != "PID"):
        if(controlador == "PI"):
            PI = True
        elif(controlador == "PD"):
            PD = True
        elif(controlador == "PID"):
            PID = True
    return PI,PD,PID


In [26]:
def definir_problema():
    metodo_1 = False
    metodo_2 = False
    metodo_3 = False
    print("Digite 1 para Mp e Epsilon.")
    print("Digite 2 para fator de amortecimento e frequência natural de amortecimento.")
    print("Digite 3 para localização de polos de malha")
    especificacoes = 0
    while(especificacoes < 1 or especificacoes > 3):
        especificacoes = int(input("Digite o número: "))
        if(especificacoes == 1):
            metodo_1 = True
        elif(especificacoes == 2):
            metodo_2 = True
        elif(especificacoes == 3):
            metodo_3 = True
    return metodo_1, metodo_2, metodo_3


In [27]:
def definir_polos_dominantes(metodo1, metodo2, metodo3):
    if(metodo1):
        Mp = int(input("Digite o Mp máximo (%): "))
        t_acomodacao = int(input("Digite o tempo de acomodação (s):"))
        tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        while(tol != 2 and tol != 5):
            tol = int(input("Digite 2 ou 5 para a tolerância (%): "))

        # Cálculo do epsilon e frequência natural
        epsilon = math.sqrt(math.log(Mp/100)**2/(math.pi**2 + math.log(Mp/100)**2))
        if(tol == 2):
            w_n = 4/(t_acomodacao*epsilon)
        elif(tol == 5):
            w_n = 3/(t_acomodacao*epsilon)

        w_d = w_n*math.sqrt(epsilon**2 +1)   
        sigma = - epsilon*w_n

        polo_1 = sigma + w_d
        polo_2 = sigma - w_d   
    if(metodo2):
        epsilon = int(input("Digite o valor do epsilon: "))
        w_n = int(input("Digite o valor da frequência natural de oscilação: "))
        w_d = w_n*math.sqrt(epsilon**2 +1)  
        polo_1 = sigma + w_d
        polo_2 = sigma - w_d    
    if(metodo3):
        polo_1 = complex(input("Digite o primeiro polo: "))
        polo_2 = complex(input("Digite o segundo polo: "))
    return polo_1, polo_2, w_d, sigma


In [28]:
def angulo(v):
    """Retorna o argumento de um número complexo em graus."""
    return float(sp.deg(sp.arg(v)))


In [48]:
def angulo_chegada(zero, polos, zeros):
    """
    Calcula o ângulo de chegada em um zero complexo.

    zero  : zero onde o ramo chega
    polos : lista com todos os polos
    zeros : lista com todos os zeros

    Retorna o ângulo em graus, no intervalo [-180, 180).
    """

    # Soma dos ângulos dos polos
    soma_polos = 0

    for p in polos:
        soma_polos += angulo(zero - p)

    # Soma dos ângulos dos outros zeros
    soma_zeros = 0

    for z in zeros:
        if z != zero:
            soma_zeros += angulo(zero - z)

    # Condição de ângulo:
    # (phi + soma_zeros) - soma_polos = 180°
    phi = 180 + soma_polos - soma_zeros

    # Normaliza para [0, 360)diferenca_polos = np.empty((0,2))
    # Normaliza para [0, 360)
    phi = phi % 360

    return phi

## Traduzir as especificações de desempenho

In [30]:
PI, PD, PID = definir_controlador()
metodo_1, metodo_2, metodo_3 = definir_problema()
polo_1, polo_2, w_d, sigma = definir_polos_dominantes(metodo_1, metodo_2, metodo_3)

Digite 1 para Mp e Epsilon.
Digite 2 para fator de amortecimento e frequência natural de amortecimento.
Digite 3 para localização de polos de malha


In [31]:
print("Forneça os coeficientes da função G(s)...")
numerador_G = polinomio(input("Digite os coeficientes do numerador: "))
denominador_G = polinomio(input("Digite os coeficientes do denominador: "))

numerador_G = sp.parse_expr(numerador_G, local_dict={'s': s})
denominador_G = sp.parse_expr(denominador_G, local_dict={'s': s})

G = TransferFunction(numerador_G, denominador_G, s)
G

Forneça os coeficientes da função G(s)...


TransferFunction(2, s**2, s)

In [32]:
print("Forneça os coeficientes da função H(s)...")
numerador_H = polinomio(input("Digite os coeficientes do numerador: "))
denominador_H = polinomio(input("Digite os coeficientes do denominador: "))

numerador_H = sp.parse_expr(numerador_H, local_dict={'s': s})
denominador_H = sp.parse_expr(denominador_H, local_dict={'s': s})

H = TransferFunction(numerador_H, denominador_H, s)
H

Forneça os coeficientes da função H(s)...


TransferFunction(1, 1, s)

In [33]:
ft = Series(G, H).doit()
ft

TransferFunction(2, s**2, s)

In [43]:
zeros, polos = pole_zero_numerical_data(ft)
print(f"Polos: {polos}")
print(f"Zeros: {zeros}")

Polos: [0.0, 0.0]
Zeros: []


In [ ]:
polo_1 = -1 + 1.95*Ipolo_1 = -1 + 1.95*I
polo_2 = -1 - 1.95*I

polos.append(polo_1)
polos.append(polo_2)
print(f"Polos: {polos}")
print(f"Zeros: {zeros}")

Polos: [0.0, 0.0, -1 + 1.95*I, -1 - 1.95*I]
Zeros: []


In [45]:
# Verifica se a função de transferência tem naturalmente um ganho
num = ft.num
den = ft.den

ganho = Poly(num, s).LC() / Poly(den, s).LC()
print(f"Ganho = {ganho}")

Ganho = 2


## PD

In [49]:
zero_temp = -1

angulo_zero = angulo_chegada(zero_temp, polos, zeros)
print(angulo_zero)

180.0


In [52]:
z = - sigma - w_d * math.pi/(math.tan(180 - angulo_zero) *180)
print(z)

ZeroDivisionError: float division by zero